# One-DM — optional fine-tune (GPU)

Optional fine-tune — **the same notebook works on Colab and a local GPU PC.**
`training/finetune.py` detects the environment, resumes from the last checkpoint
(Google Drive on Colab, local disk on a PC) and backs up every epoch.

Only needed if the pretrained One-DM visibly fails on your style. Requires the official checkpoint + English data bundle and `vae_HTR138.pth`.

CPU-only machine? Training is refused with instructions — use the quick test
(`00_quick_test.ipynb`) instead.

## 0. Where am I running?

In [ ]:
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content/.config")
print("Running on:", "Google Colab" if IN_COLAB else "local machine (GPU PC or CPU laptop)")

## 1. Checkpoints (Drive on Colab / local disk on a GPU PC)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("checkpoints will resume/backup to Drive automatically")
else:
    print("local run: checkpoints stay on this machine (same commands)")

## 2. Get the project code here

In [ ]:
REPO_URL = "https://github.com/Jeevant010/Text_Writter"
if IN_COLAB:
    root = pathlib.Path("/content/Text_Writter")
    if not (root / "src/textwritter/quicktest.py").exists():
        !git clone --depth 1 {REPO_URL} {root}
    os.chdir(root)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    os.chdir(root)
print("repo root:", pathlib.Path.cwd())

## 3. Dependencies

In [ ]:
missing = []
for mod in ["torch", "torchvision", "cv2", "PIL", "transformers", "numpy", "matplotlib", "gdown"]:
    try:
        __import__(mod)
    except Exception:
        missing.append(mod)
if missing:
    print("installing:", missing)
    !{sys.executable} -m pip install -q -r requirements.txt
else:
    print("dependencies already available")

## 4. Check prerequisites (no training yet)

In [ ]:
import subprocess
print("exit:", subprocess.run([sys.executable, "training/finetune.py",
      "--model", "onedm", "--check"]).returncode)

## 5. Your handwriting data

A zip containing cropped line/word images plus `labels.txt`
(one `file.png<TAB>text` line per image). 10–30 lines is plenty.

In [ ]:
DATA = None
if IN_COLAB:
    from google.colab import files
    print("Upload my_handwriting.zip")
    up = files.upload()
    if up:
        name = list(up)[0]
        pathlib.Path("data_uploads").mkdir(exist_ok=True)
        pathlib.Path(name).rename(root / "data_uploads" / name)
        DATA = str(root / "data_uploads" / name)
else:
    import glob
    cands = sorted(glob.glob("samples/*.zip") + glob.glob("data_uploads/*.zip"))
    DATA = cands[0] if cands else None
print("data:", DATA or "none -> put a zip in samples/ (local) or upload it (Colab)")

## 6. Fine-tune (interrupt-safe: re-run this cell to resume)

In [ ]:
import subprocess
if not DATA:
    print("No data zip found — see the previous cell.")
else:
    cmd = [sys.executable, "training/finetune.py", "--model", "onedm",
           "--data", DATA, "--epochs", "15"]
    print(" ".join(cmd))
    print("exit:", subprocess.run(cmd).returncode)

## 7. Done

The newest checkpoint is in `checkpoints/<run-name>/` (or on Google Drive under
`MyDrive/textwritter_checkpoints/<run-name>/`). Point inference at it, or just
re-run the quick test — nothing is trained from scratch, ever.